# 01 — Exploratory Data Analysis (Store Sales)

**Goal of this notebook:** understand the data well enough that every later modeling choice is *motivated by evidence*, not guessed. It produces no model — it produces insight and verified data integrity.

We read the raw competition CSVs through the small `src/data.py` loaders (so the notebook stays thin and the loading logic is reused and tested in one place), then work through the data in sections: shapes & integrity first, then trend, seasonality, calendar/holidays, promotions/oil, and anomalies.

Run it top-to-bottom from a fresh kernel.

## Section 1 — Load the data & confirm what we're working with

Before any plot or model we answer one question: **is the data exactly what every later step assumes?**

We load all seven CSVs, print each one's shape and (where it has a `date`) its date range, then *assert* the four headline facts of this competition:

- **54** stores
- **33** product families
- **54 × 33 = 1,782** time series (one per store × family)
- training history spans **2013-01-01 → 2017-08-15**

Why assert and not just print? A printed number you read once; an `assert` checks the number on *every* run and fails loudly the moment the data changes underneath us. That turns a silent corruption (a re-downloaded CSV with a different shape) into an obvious error instead of a wrong model.

In [1]:
import sys
from pathlib import Path

# The kernel's working directory is `notebooks/`, but our reusable code lives in the
# project's `src/` package one level up. Put the repo root on the import path so
# `import src.data` works whether the kernel started in the repo root or in notebooks/.
here = Path.cwd()
repo_root = here if (here / "src").exists() else here.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import src.data as data

print("repo root:", repo_root)

repo root: D:\AIO\M01\Conquer\TimeSeries


In [2]:
# Load every raw CSV through the loaders in src/data.py (read-only source of truth).
train = data.load_train()
test = data.load_test()
stores = data.load_stores()
transactions = data.load_transactions()
oil = data.load_oil()
holidays = data.load_holidays()
sample_submission = data.load_sample_submission()

# Shape of each table; date range for the ones that carry a calendar.
frames = {
    "train": train,
    "test": test,
    "stores": stores,
    "transactions": transactions,
    "oil": oil,
    "holidays": holidays,
    "sample_submission": sample_submission,
}

for name, df in frames.items():
    if "date" in df.columns:
        span = f"{df['date'].min().date()} -> {df['date'].max().date()}"
    else:
        span = "(no date column)"
    print(f"{name:18s} shape={str(df.shape):16s} {span}")

train              shape=(3000888, 6)     2013-01-01 -> 2017-08-15
test               shape=(28512, 5)       2017-08-16 -> 2017-08-31
stores             shape=(54, 5)          (no date column)
transactions       shape=(83488, 3)       2013-01-01 -> 2017-08-15
oil                shape=(1218, 2)        2013-01-01 -> 2017-08-31
holidays           shape=(350, 6)         2012-03-02 -> 2017-12-26
sample_submission  shape=(28512, 2)       (no date column)


In [3]:
# Confirm the four headline facts. Each assert fails loudly if the data ever shifts;
# the print line lets us read the numbers we just checked.
n_stores = stores["store_nbr"].nunique()
n_families = train["family"].nunique()
n_series = train.groupby(["store_nbr", "family"], observed=True).ngroups
train_start = train["date"].min().date()
train_end = train["date"].max().date()

assert n_stores == 54, f"expected 54 stores, got {n_stores}"
assert n_families == 33, f"expected 33 families, got {n_families}"
assert n_series == 1782, f"expected 1782 series (54x33), got {n_series}"
assert str(train_start) == "2013-01-01", f"unexpected train start {train_start}"
assert str(train_end) == "2017-08-15", f"unexpected train end {train_end}"

print(f"stores   : {n_stores}")
print(f"families : {n_families}")
print(f"series   : {n_series}  (= {n_stores} x {n_families})")
print(f"train    : {train_start} -> {train_end}")
print("\nAll headline counts confirmed.")

stores   : 54
families : 33
series   : 1782  (= 54 x 33)
train    : 2013-01-01 -> 2017-08-15

All headline counts confirmed.


**What this tells us / what's next:** the data is intact and matches expectations — 1,782 daily series over ~4.6 years, with a fixed 16-day forecast horizon (test: 2017-08-16 → 2017-08-31). One caveat the shapes already hint at: `train` has far fewer rows than `1782 series × every calendar day`, because the store is closed on a few days (e.g. Dec 25) and those rows are simply *absent* rather than recorded as zero. The next section restores a complete daily calendar per series so that lag and seasonality features count days correctly.